In [29]:
from nilearn import plotting
%matplotlib inline
from os.path import join as opj
import json
from nipype.interfaces.base import Bunch
from nipype.interfaces.spm import Level1Design, EstimateModel, EstimateContrast, SPMCommand, Info, model
from nipype.interfaces.matlab import MatlabCommand
from nipype.interfaces.freesurfer import FSCommand
from nipype.algorithms.modelgen import SpecifySPMModel, SpecifyModel
from nipype.interfaces.utility import Function, IdentityInterface
from nipype.interfaces.io import SelectFiles, DataSink
from nipype import Workflow, Node
from bids.layout import BIDSLayout
from glob import glob
from scipy import io, stats
from itertools import chain
import pandas as pd
import numpy as np
#import pytest as pt # not even needed
import nibabel as nb
import nipype
import os.path as op
from nipype.interfaces import spm

MatlabCommand.set_default_matlab_cmd('/Applications/MATLAB_R2021b.app/bin/matlab')#'/Users/mrenke/matlab') # /Users/mrenke/matlab
bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'

# for local
spm.SPMCommand.set_mlab_paths(matlab_cmd="/Applications/MATLAB_R2021b.app/bin/matlab") # error during spm.EstimateModel, solved: https://neurostars.org/t/nipype-spm-spmcommand-version-does-not-return-anything-on-mac/2871

# for sciencecloud2
MatlabCommand.set_default_paths('/home/ubuntu/matlab/spm12') # 

print(spm.SPMCommand().version)

/Users/mrenke/mambaforge/envs/numrefields/lib/python3.10/site-packages/nipype/__init__.py


stty: stdin isn't a terminal


12.7771


stty: stdin isn't a terminal


In [30]:
sub='01'
ses = 1
run = 1

with open(op.join(bids_folder, 'sub-01', 'ses-1','func', 'sub-01_ses-1_task-risk_run-1_bold.json'), # 'derivatives/fmriprep' , 
    "rt",
) as fp:
    task_info = json.load(fp)
TR = task_info["RepetitionTime"]
Nslices = len(task_info['SliceTiming']) # = 39
refSlice = 20 #Nslices / 2

In [31]:
def get_subject_info(subject):
    from glob import glob
    import numpy as np
    import pandas as pd
    from scipy import io, stats
    from nipype.interfaces.base import Bunch
    import os.path as op
    bids_folder = '/Volumes/mrenkeED/data/ds-stressrisk'
    ses=1
    sub = subject
    subject_info = []

    functional_runs = []
    
    for ses in [1,2]:
        for run in range(1, 7):
            run_ses_i = (ses - 1)*6 + run
            
            # regressors (confounds + physio)
            confounds = pd.read_csv(op.join(bids_folder, 'derivatives/fmriprep',f'sub-{sub}', f'ses-{ses}', 'func', 
                            f'sub-{sub}_ses-{ses}_task-risk_run-{run}_desc-confounds_timeseries.tsv'), sep='\t')
            confound_names = ["trans_x","trans_y","trans_z","rot_x","rot_y","rot_z","a_comp_cor_00","a_comp_cor_02","a_comp_cor_03","a_comp_cor_04"]
            confounds = confounds.loc[:, confound_names]
            fn_physio = op.join(bids_folder, 'derivatives/physiotoolbox',f'sub-{sub}', f'ses-{ses}', 'func', 
                                f'sub-{sub}_ses-{ses}_task-task_run-{run}_desc-retroicor_output.mat') # task-taks (not risk) 
            physio = io.loadmat(fn_physio, simplify_cells=True)["physio"]["model"]
            physio = pd.DataFrame(
                data=physio["R"],
                columns=physio["R_column_names"])
            regressors = pd.concat([confounds, physio], axis=1)
            regressor_names = regressors.columns.values.tolist()

            # events + pmods
            df_events = pd.read_csv(op.join(bids_folder, f'sub-{sub}', f'ses-{ses}', 'func', f'sub-{sub}_ses-{ses}_task-risk_run-{run}_events.tsv'), sep='\t')
            df_events.set_index(['trial_nr', 'trial_type'], inplace=True) # only look at second option
            
            df_events_num1 = df_events.xs('stimulus 1',0,'trial_type')[['onset','prob1','n1']]
            df_risky_num1 = df_events_num1[df_events_num1['prob1'] == 0.55]
            df_safe_num1 = df_events_num1[df_events_num1['prob1'] == 1]   
            
            df_events_num2 = df_events.xs('stimulus 2',0,'trial_type')[['onset','prob2','n2']]
            df_risky_num2 = df_events_num2[df_events_num2['prob2'] == 0.55]
            df_safe_num2 = df_events_num2[df_events_num2['prob2'] == 1]

            pmod = [
                Bunch(name=['num1_risky'], param=[df_risky_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num1_safe'],param=[df_safe_num1['n1'].values.tolist()], poly=[1]),
                Bunch(name=['num2_risky'], param=[df_risky_num2['n2'].values.tolist()], poly=[1]),
                Bunch(name=['num2_safe'],param=[df_safe_num2['n2'].values.tolist()], poly=[1])]
            onsets = [
                df_risky_num1['onset'].values.tolist(), 
                df_safe_num1['onset'].values.tolist(),
                df_risky_num2['onset'].values.tolist(), 
                df_safe_num2['onset'].values.tolist()]

            durations = [(np.ones(len(df_risky_num1))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num1))* 0.6).tolist(),
                         (np.ones(len(df_risky_num2))* 0.6).tolist(), 
                         (np.ones(len(df_safe_num2))* 0.6).tolist()]

            # put all together
            conditions = [f'risky_num1_ses{ses}', f'safe_num1_ses{ses}', f'risky_num2_ses{ses}', f'safe_num2_ses{ses}']
            subject_info.insert(
                run_ses_i - 1,
                Bunch(
                    conditions=conditions,
                    onsets=onsets,
                    durations=durations,
                    pmod=pmod,
                    tmod=None,
                    orth=['No']*len(conditions),
                    regressors=regressors.values.T.tolist(),
                    regressor_names=regressor_names,
                ),
            )

            #nifit_file = op.join(bids_folder,'derivatives/fmriprep',f'sub-{sub}', f'ses-{ses}', 'func',)
            nifti_file =  op.join(bids_folder,'derivatives/spm_nipype', f'sub-{sub}', f'ses-{ses}', f'ssub-{sub}_ses-{ses}_task-risk_run-{run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii')
            functional_run = glob(nifti_file)[0]
            functional_runs.append(functional_run)

    return subject_info, functional_runs

# try function
sub = '01'
subject_info, functional_runs = get_subject_info(sub)
functional_runs

['/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-2_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-3_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-4_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-5_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/ses-1/ssub-01_ses-1_task-risk_run-6_space-MNI152NLin2009cAsym_desc-preproc_bold.nii',
 '/Volumes/mrenkeED/data/ds-stressrisk/derivatives/spm_nipype/sub-01/s

In [35]:
def get_contrasts(subject_info):
    from nipype.interfaces.spm import EstimateContrast
    import os

    pmod_names = ['risky_num2_ses1xnum2_risky^1','risky_num2_ses2xnum2_risky^1', # 'group by riksy/safe for easier indexiing
                'safe_num2_ses1xnum2_safe^1','safe_num2_ses2xnum2_safe^1']
    condition_names = ['risky_num2_ses1','risky_num2_ses2','safe_num2_ses1','safe_num2_ses2']
    
    # same for both sessions
    con01 = ('num2_risky_int_bothSes', 'T', [condition_names[0:2]], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con02 = ('num2_safe_int_bothSes', 'T', [condition_names[2:4]], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con03 = ('num2_risky_pmod_bothSes', 'T', [pmod_names[0:2]], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con04 = ('num2_safe_pmod_bothSes', 'T', [pmod_names[2:4]], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
    
    # difference between sessions
    con05 = ('num2_risky_int_sesDif', 'T', [condition_names[0:2]], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con06 = ('num2_safe_int_sesDif', 'T', [condition_names[2:4]], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con07 = ('num2_risky_sesDif', 'T', [pmod_names[0:2]], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con08 = ('num2_safe_sesDif', 'T', [pmod_names[2:4]], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
       
    con_list = [con01, con02, con03, con04, con05,con06, con07, con08]
    
    return con_list


### start the workflow with runnning single nodes

In [39]:

getsubjectinfo = Node(
    Function(
        input_names=["subject"],
        output_names=["subject_info", "functional_runs"],
        function=get_subject_info,
    ),
    name="getsubjectinfo",
)

getsubjectinfo.inputs.subject = '01'
getsubjectinfo = getsubjectinfo.run()

240307-14:00:57,527 nipype.workflow INFO:
	 [Node] Setting-up "getsubjectinfo" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpnpve5lcj/getsubjectinfo".
240307-14:00:57,532 nipype.workflow INFO:
	 [Node] Executing "getsubjectinfo" <nipype.interfaces.utility.wrappers.Function>
240307-14:00:58,305 nipype.workflow INFO:
	 [Node] Finished "getsubjectinfo", elapsed time 0.771929s.


In [41]:
modelspec = Node(
    SpecifySPMModel(
        concatenate_runs=False,
        input_units="secs",
        output_units="secs", # 'scans' ??
        time_repetition=TR,
        high_pass_filter_cutoff=128,
    ),
    name="modelspec",
)

modelspec.inputs.subject_info = getsubjectinfo.outputs.subject_info
modelspec.inputs.functional_runs = getsubjectinfo.outputs.functional_runs

modelspec = modelspec.run()
print(modelspec.outputs.session_info)

240307-14:01:15,57 nipype.workflow INFO:
	 [Node] Setting-up "modelspec" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmp0tswvb74/modelspec".
240307-14:01:15,213 nipype.workflow INFO:
	 [Node] Executing "modelspec" <nipype.algorithms.modelgen.SpecifySPMModel>
240307-14:01:15,218 nipype.workflow INFO:
	 [Node] Finished "modelspec", elapsed time 0.002877s.
[{'cond': [{'name': 'risky_num1_ses1', 'onset': [13.26218560000052, 25.923746000000374, 42.072033699998414, 57.219389999998384, 70.3649035999988, 82.5094691000013, 99.15822610000032, 113.82175629999983, 129.4695431, 142.61511489999972], 'duration': [0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6, 0.6], 'pmod': [{'name': 'num1_risky', 'poly': 1, 'param': [13.0, 8.0, 13.0, 38.0, 16.0, 32.0, 46.0, 46.0, 55.0, 55.0]}]}, {'name': 'safe_num1_ses1', 'onset': [162.767042200001, 178.4149241999985, 191.56041090000144, 207.20821440000145, 219.36945479999844, 234.01638939999972, 249.6641367000012, 264.8115929000014, 279.4584461000013,

In [42]:
# Level1Design - Generates an SPM design matrix
level1design = Node(
    Level1Design(
        bases={"hrf": {"derivs": [0, 0]}},
        timing_units="secs",
        interscan_interval=TR,
        model_serial_correlations="AR(1)",
        microtime_resolution=Nslices,
        microtime_onset=refSlice,
        flags={"mthresh": 0.8, "globalnorm": 'None'},
        #mask_image="/mnt/d/multlearn-sns/SPM/mask_ICV.nii", ??
        volterra_expansion_order=1,
    ),
    name="level1design",
)

level1design.inputs.session_info = modelspec.outputs.session_info
level1design = level1design.run()
print(level1design.outputs.spm_mat_file)

240307-14:01:45,376 nipype.workflow INFO:
	 [Node] Setting-up "level1design" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmp03erw6md/level1design".
240307-14:01:45,774 nipype.workflow INFO:
	 [Node] Executing "level1design" <nipype.interfaces.spm.model.Level1Design>


stty: stdin isn't a terminal


240307-14:03:51,558 nipype.workflow INFO:
	 [Node] Finished "level1design", elapsed time 113.765274s.
/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmp03erw6md/level1design/SPM.mat


stty: stdin isn't a terminal


In [50]:
# EstimateModel - estimate the parameters of the model
level1estimate = Node( EstimateModel(estimation_method={"Classical": 1},write_residuals=False), name="level1estimate")

level1estimate.inputs.spm_mat_file = level1design.outputs.spm_mat_file
level1estimate = level1estimate.run()

240307-14:15:04,659 nipype.workflow INFO:
	 [Node] Setting-up "level1estimate" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmp290j6jyw/level1estimate".
240307-14:15:04,690 nipype.workflow INFO:
	 [Node] Executing "level1estimate" <nipype.interfaces.spm.model.EstimateModel>


stty: stdin isn't a terminal
stty: stdin isn't a terminal


240307-14:20:23,73 nipype.workflow INFO:
	 [Node] Finished "level1estimate", elapsed time 306.017551s.


In [51]:
def get_contrasts(subject_info):
    from nipype.interfaces.spm import EstimateContrast
    import os

    pmod_names = ['risky_num2_ses1xnum2_risky^1','risky_num2_ses2xnum2_risky^1', # 'group by riksy/safe for easier indexiing
                'safe_num2_ses1xnum2_safe^1','safe_num2_ses2xnum2_safe^1']
    condition_names = ['risky_num2_ses1','risky_num2_ses2','safe_num2_ses1','safe_num2_ses2']
    
    # same for both sessions
    con01 = ('num2_risky_int_bothSes', 'T', condition_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con02 = ('num2_safe_int_bothSes', 'T', condition_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con03 = ('num2_risky_pmod_bothSes', 'T', pmod_names[0:2], [1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con04 = ('num2_safe_pmod_bothSes', 'T', pmod_names[2:4], [1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
    
    # difference between sessions
    con05 = ('num2_risky_int_sesDif', 'T', condition_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con06 = ('num2_safe_int_sesDif', 'T', condition_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]  
    con07 = ('num2_risky_sesDif', 'T', pmod_names[0:2], [-1/2. , 1/2.]) #['num2_risky', 'T', [condition_names[0]], [1.]]
    con08 = ('num2_safe_sesDif', 'T', pmod_names[2:4], [-1/2. , 1/2.])#['num2_safe', 'T', [condition_names[1]], [1.]]
       
    con_list = [con01, con02, con03, con04, con05,con06, con07, con08]
    
    return con_list

In [52]:
# EstimateContrast - estimates contrasts
getcontrasts = Node(Function(
        input_names=["subject_info"],
        output_names=["contrasts"],
        function=get_contrasts),
        name="getcontrasts")

getcontrasts.inputs.subject_info = getsubjectinfo.outputs.subject_info # output from very first node
getcontrasts = getcontrasts.run()

level1conest = Node(EstimateContrast(), name="level1conest")
level1conest.inputs.contrasts =  getcontrasts.outputs.contrasts
level1conest.inputs.spm_mat_file = level1estimate.outputs.spm_mat_file
level1conest.inputs.beta_images = level1estimate.outputs.beta_images
level1conest.inputs.residual_image = level1estimate.outputs.residual_image
level1conest = level1conest.run()
print(level1conest.outputs)

240307-14:25:42,306 nipype.workflow INFO:
	 [Node] Setting-up "getcontrasts" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmp4pflx84c/getcontrasts".


240307-14:25:42,468 nipype.workflow INFO:
	 [Node] Executing "getcontrasts" <nipype.interfaces.utility.wrappers.Function>
240307-14:25:42,470 nipype.workflow INFO:
	 [Node] Finished "getcontrasts", elapsed time 0.000451s.
240307-14:25:42,548 nipype.workflow INFO:
	 [Node] Setting-up "level1conest" in "/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest".
240307-14:25:42,684 nipype.workflow INFO:
	 [Node] Executing "level1conest" <nipype.interfaces.spm.model.EstimateContrast>


stty: stdin isn't a terminal
stty: stdin isn't a terminal


240307-14:26:21,629 nipype.workflow INFO:
	 [Node] Finished "level1conest", elapsed time 26.032684s.

con_images = ['/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0001.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0002.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0003.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0004.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0005.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0006.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0007.nii', '/private/var/folders/3k/8g0xv78x051fznwyh_m5xcn8f91w3q/T/tmpvh3c9vik/level1conest/con_0008.nii']
ess_images = <undefined>
spmF_images = <undefined>
spmT_images = ['/private/var/folders/3k/8g0xv78x0